In [1]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded.keys():
    print(filename)

Saving IT_knowledge_base.csv to IT_knowledge_base.csv
Saving it_ticket_cluster_report.csv to it_ticket_cluster_report.csv

Uploaded files:
IT_knowledge_base.csv
it_ticket_cluster_report.csv


In [2]:
import os

print("Checking files...\n")

print("it_ticket_cluster_report.csv:",
      os.path.exists("it_ticket_cluster_report.csv"))

print("IT_knowledge_base.csv:",
      os.path.exists("IT_knowledge_base.csv"))

Checking files...

it_ticket_cluster_report.csv: True
IT_knowledge_base.csv: True


In [3]:
import pandas as pd

cluster_df = pd.read_csv("it_ticket_cluster_report.csv")
knowledge_df = pd.read_csv("IT_knowledge_base.csv")

print("Ticket Cluster Report loaded successfully.")
print("Knowledge Base loaded successfully.")

Ticket Cluster Report loaded successfully.
Knowledge Base loaded successfully.


In [4]:
print("===== TICKET CLUSTER REPORT =====")
display(cluster_df.head())

print("\n===== IT KNOWLEDGE BASE =====")
display(knowledge_df.head())

===== TICKET CLUSTER REPORT =====


,ticket_id,issue,category,resolution_time_hours,resolution_class,category_code,cluster
0,1,Password reset required,Account,1.2,Fast,0,0
1,2,Unable to login,Account,2.5,Fast,0,0
2,3,WiFi connection not working,Network,5.5,Medium,1,0
3,4,Printer not responding,Hardware,7.2,Medium,2,0
4,5,Software installation problem,Software,10.5,Slow,3,2



===== IT KNOWLEDGE BASE =====


,title,problem,solution
0,Password Reset,User forgot their password or cannot access th...,"Open the company password reset portal, enter ..."
1,WiFi Connection Problem,Computer cannot connect to the office WiFi net...,Check that WiFi is enabled. Disconnect and rec...
2,VPN Connection Failure,User cannot connect to the company VPN.,"Check your internet connection, restart the VP..."
3,Printer Problem,Printer is offline or documents are not printing.,Check printer power and network connection. Ma...
4,Email Not Working,User cannot send or receive emails.,"Check the internet connection, verify mailbox ..."


In [5]:
# Make sure text columns contain no empty values

cluster_df["issue"] = cluster_df["issue"].fillna("").astype(str)
knowledge_df["problem"] = knowledge_df["problem"].fillna("").astype(str)

print("Ticket issues prepared:", len(cluster_df))
print("Knowledge-base problems prepared:", len(knowledge_df))

Ticket issues prepared: 20
Knowledge-base problems prepared: 8


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [7]:
# Combine ticket issues and historical problems
all_text = (
    cluster_df["issue"].tolist()
    + knowledge_df["problem"].tolist()
)

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

# Convert text into numerical vectors
tfidf_matrix = vectorizer.fit_transform(all_text)

# Separate ticket vectors and knowledge-base vectors
ticket_vectors = tfidf_matrix[:len(cluster_df)]
knowledge_vectors = tfidf_matrix[len(cluster_df):]

print("TF-IDF conversion completed.")
print("Ticket vectors:", ticket_vectors.shape)
print("Knowledge-base vectors:", knowledge_vectors.shape)

TF-IDF conversion completed.
Ticket vectors: (20, 87)
Knowledge-base vectors: (8, 87)


In [8]:
# Calculate similarity between every ticket and every historical problem
similarity_matrix = cosine_similarity(
    ticket_vectors,
    knowledge_vectors
)

print("Similarity calculation completed.")
print("Similarity matrix shape:", similarity_matrix.shape)

Similarity calculation completed.
Similarity matrix shape: (20, 8)


In [9]:
# Find the index of the most similar historical problem for each ticket
best_match_index = similarity_matrix.argmax(axis=1)

# Get the similarity score of the best match
best_similarity_score = similarity_matrix.max(axis=1)

print("Best historical matches identified.")
print("Number of matched tickets:", len(best_match_index))

Best historical matches identified.
Number of matched tickets: 20


In [10]:
# Add the matched historical knowledge to each ticket

cluster_df["matched_title"] = [
    knowledge_df.iloc[i]["title"]
    for i in best_match_index
]

cluster_df["matched_problem"] = [
    knowledge_df.iloc[i]["problem"]
    for i in best_match_index
]

cluster_df["historical_solution"] = [
    knowledge_df.iloc[i]["solution"]
    for i in best_match_index
]

cluster_df["similarity_score"] = best_similarity_score

print("Historical solutions connected successfully.")

Historical solutions connected successfully.


In [11]:
final_mapping = cluster_df[
    [
        "ticket_id",
        "issue",
        "category",
        "cluster",
        "matched_title",
        "matched_problem",
        "historical_solution",
        "similarity_score"
    ]
].copy()

print("Final ticket-to-solution mapping created.")
display(final_mapping.head(10))

Final ticket-to-solution mapping created.


,ticket_id,issue,category,cluster,matched_title,matched_problem,historical_solution,similarity_score
0,1,Password reset required,Account,0,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141304
1,2,Unable to login,Account,0,Password Reset,User forgot their password or cannot access th...,"Open the company password reset portal, enter ...",0.000000
2,3,WiFi connection not working,Network,0,WiFi Connection Problem,Computer cannot connect to the office WiFi net...,Check that WiFi is enabled. Disconnect and rec...,0.127286
3,4,Printer not responding,Hardware,0,Printer Problem,Printer is offline or documents are not printing.,Check printer power and network connection. Ma...,0.170407
4,5,Software installation problem,Software,2,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141304
5,6,Email not working,Email,1,Password Reset,User forgot their password or cannot access th...,"Open the company password reset portal, enter ...",0.000000
6,7,Computer running slow,Performance,2,Computer Running Slow,Computer performance is very slow.,"Close unnecessary applications, restart the co...",0.300637
7,8,VPN connection failed,Network,0,VPN Connection Failure,User cannot connect to the company VPN.,"Check your internet connection, restart the VP...",0.144258
8,9,Password reset required,Account,0,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141304
9,10,Blue screen error,Hardware,2,Blue Screen Error,Computer displays a blue screen and restarts u...,Restart the computer and record the error mess...,0.340294


In [12]:
def get_confidence(score):
    if score >= 0.70:
        return "HIGH"
    elif score >= 0.40:
        return "MEDIUM"
    else:
        return "LOW"

final_mapping["match_confidence"] = (
    final_mapping["similarity_score"]
    .apply(get_confidence)
)

display(final_mapping.head(10))

,ticket_id,issue,category,cluster,matched_title,matched_problem,historical_solution,similarity_score,match_confidence
0,1,Password reset required,Account,0,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141304,LOW
1,2,Unable to login,Account,0,Password Reset,User forgot their password or cannot access th...,"Open the company password reset portal, enter ...",0.000000,LOW
2,3,WiFi connection not working,Network,0,WiFi Connection Problem,Computer cannot connect to the office WiFi net...,Check that WiFi is enabled. Disconnect and rec...,0.127286,LOW
3,4,Printer not responding,Hardware,0,Printer Problem,Printer is offline or documents are not printing.,Check printer power and network connection. Ma...,0.170407,LOW
4,5,Software installation problem,Software,2,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141304,LOW
5,6,Email not working,Email,1,Password Reset,User forgot their password or cannot access th...,"Open the company password reset portal, enter ...",0.000000,LOW
6,7,Computer running slow,Performance,2,Computer Running Slow,Computer performance is very slow.,"Close unnecessary applications, restart the co...",0.300637,LOW
7,8,VPN connection failed,Network,0,VPN Connection Failure,User cannot connect to the company VPN.,"Check your internet connection, restart the VP...",0.144258,LOW
8,9,Password reset required,Account,0,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141304,LOW
9,10,Blue screen error,Hardware,2,Blue Screen Error,Computer displays a blue screen and restarts u...,Restart the computer and record the error mess...,0.340294,LOW


In [13]:
final_mapping["similarity_score"] = (
    final_mapping["similarity_score"].round(3)
)

display(final_mapping.head(10))

,ticket_id,issue,category,cluster,matched_title,matched_problem,historical_solution,similarity_score,match_confidence
0,1,Password reset required,Account,0,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141,LOW
1,2,Unable to login,Account,0,Password Reset,User forgot their password or cannot access th...,"Open the company password reset portal, enter ...",0.000,LOW
2,3,WiFi connection not working,Network,0,WiFi Connection Problem,Computer cannot connect to the office WiFi net...,Check that WiFi is enabled. Disconnect and rec...,0.127,LOW
3,4,Printer not responding,Hardware,0,Printer Problem,Printer is offline or documents are not printing.,Check printer power and network connection. Ma...,0.170,LOW
4,5,Software installation problem,Software,2,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141,LOW
5,6,Email not working,Email,1,Password Reset,User forgot their password or cannot access th...,"Open the company password reset portal, enter ...",0.000,LOW
6,7,Computer running slow,Performance,2,Computer Running Slow,Computer performance is very slow.,"Close unnecessary applications, restart the co...",0.301,LOW
7,8,VPN connection failed,Network,0,VPN Connection Failure,User cannot connect to the company VPN.,"Check your internet connection, restart the VP...",0.144,LOW
8,9,Password reset required,Account,0,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141,LOW
9,10,Blue screen error,Hardware,2,Blue Screen Error,Computer displays a blue screen and restarts u...,Restart the computer and record the error mess...,0.340,LOW


In [14]:
print("===== COMPLETE TICKET-SOLUTION MAPPING =====")

display(final_mapping)

===== COMPLETE TICKET-SOLUTION MAPPING =====


,ticket_id,issue,category,cluster,matched_title,matched_problem,historical_solution,similarity_score,match_confidence
0,1,Password reset required,Account,0,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141,LOW
1,2,Unable to login,Account,0,Password Reset,User forgot their password or cannot access th...,"Open the company password reset portal, enter ...",0.000,LOW
2,3,WiFi connection not working,Network,0,WiFi Connection Problem,Computer cannot connect to the office WiFi net...,Check that WiFi is enabled. Disconnect and rec...,0.127,LOW
3,4,Printer not responding,Hardware,0,Printer Problem,Printer is offline or documents are not printing.,Check printer power and network connection. Ma...,0.170,LOW
4,5,Software installation problem,Software,2,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141,LOW
5,6,Email not working,Email,1,Password Reset,User forgot their password or cannot access th...,"Open the company password reset portal, enter ...",0.000,LOW
6,7,Computer running slow,Performance,2,Computer Running Slow,Computer performance is very slow.,"Close unnecessary applications, restart the co...",0.301,LOW
7,8,VPN connection failed,Network,0,VPN Connection Failure,User cannot connect to the company VPN.,"Check your internet connection, restart the VP...",0.144,LOW
8,9,Password reset required,Account,0,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141,LOW
9,10,Blue screen error,Hardware,2,Blue Screen Error,Computer displays a blue screen and restarts u...,Restart the computer and record the error mess...,0.340,LOW


In [15]:
output_file = "ticket_cluster_historical_solution_mapping.csv"

final_mapping.to_csv(
    output_file,
    index=False
)

print("CSV created successfully:", output_file)

CSV created successfully: ticket_cluster_historical_solution_mapping.csv


In [16]:
import os

print("File exists:", os.path.exists(output_file))

if os.path.exists(output_file):
    print("File size:", os.path.getsize(output_file), "bytes")

File exists: True
File size: 5303 bytes


In [17]:
check_df = pd.read_csv(output_file)

print("Saved CSV contents:")
display(check_df.head())

Saved CSV contents:


,ticket_id,issue,category,cluster,matched_title,matched_problem,historical_solution,similarity_score,match_confidence
0,1,Password reset required,Account,0,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141,LOW
1,2,Unable to login,Account,0,Password Reset,User forgot their password or cannot access th...,"Open the company password reset portal, enter ...",0.000,LOW
2,3,WiFi connection not working,Network,0,WiFi Connection Problem,Computer cannot connect to the office WiFi net...,Check that WiFi is enabled. Disconnect and rec...,0.127,LOW
3,4,Printer not responding,Hardware,0,Printer Problem,Printer is offline or documents are not printing.,Check printer power and network connection. Ma...,0.170,LOW
4,5,Software installation problem,Software,2,Software Installation,User cannot install required software.,Verify that the software is approved by the or...,0.141,LOW


In [18]:
from google.colab import files

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
cluster_solution_summary = (
    final_mapping
    .groupby(["cluster", "matched_title"])
    .size()
    .reset_index(name="ticket_count")
)

display(cluster_solution_summary)

,cluster,matched_title,ticket_count
0,0,Password Reset,2
1,0,Printer Problem,2
2,0,Software Installation,2
3,0,VPN Connection Failure,2
4,0,WiFi Connection Problem,2
5,1,Password Reset,3
6,2,Blue Screen Error,2
7,2,Computer Running Slow,2
8,2,Password Reset,1
9,2,Software Installation,2


In [20]:
print("==============================================")
print("TICKET CLUSTER → HISTORICAL SOLUTION")
print("==============================================")

print("Total tickets:", len(final_mapping))
print("Total clusters:", final_mapping["cluster"].nunique())
print("Historical solutions:", knowledge_df["title"].nunique())

print("\nOutput file:")
print(output_file)

print("\nProject step completed successfully.")

TICKET CLUSTER → HISTORICAL SOLUTION
Total tickets: 20
Total clusters: 3
Historical solutions: 8

Output file:
ticket_cluster_historical_solution_mapping.csv

Project step completed successfully.
